# ResNet-50 Training - Separated Experiments
Each preprocessing version runs in a separate cell to avoid crashes:
- **Experiment 1**: Raw
- **Experiment 2**: Unified  
- **Experiment 3**: Customized

In [3]:
#%pip install torch torchvision
%pip install scikit-learn scikit-image
%pip install opencv-python
%pip install ultralytics
%pip install pandas numpy tqdm
%pip install pillow

  Using cached lazy_loader-0.4-py3-none-any.whl.metadata (7.6 kB)
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   --- ------------------------------------ 1.0/11.9 MB 8.5 MB/s eta 0:00:02
   ----------------- ---------------------- 5.2/11.9 MB 17.7 MB/s eta 0:00:01
   -------------------------------- ------- 9.7/11.9 MB 20.2 MB/s eta 0:00:01
   ---------------------------------------- 11.9/11.9 MB 20.2 MB/s  0:00:00
Using cached lazy_loader-0.4-py3-none-any.whl (12 kB)

   ---------------------------------------- 0/4 [tifffile]
   -------------------- ------------------- 2/4 [imageio]
   ------------------------------ --------- 3/4 [scikit-image]
   ------------------------------ --------- 3/4 [scikit-image]
   ------------------------------ --------- 3/4 [scikit-image]
   ---------------------------------------- 4/4 [scikit-image]

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import os, json, gc
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

# SETUP & DEVICE
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = r"C:\Users\YEO\OneDrive\Desktop\Degree\YEAR 3\Sem2\TCV\Project\data"
MODEL_SAVE_DIR = "."
JSON_PATH = os.path.join(DATA_DIR, "dataset_split.json")
CATEGORIES = ['Can', 'Paper', 'Plastic Bag', 'Plastic Bottle']

print(f"Using device: {DEVICE}")
print(f"Data directory: {DATA_DIR}")
print(f"Model save directory: {MODEL_SAVE_DIR}")

# Load dataset split once
with open(JSON_PATH, 'r') as f: 
    split = json.load(f)
    
print(f"Train samples: {len(split['train'])}")
print(f"Test samples: {len(split['test'])}")

Using device: cuda
Data directory: C:\Users\YEO\OneDrive\Desktop\Degree\YEAR 3\Sem2\TCV\Project\data
Model save directory: .
Train samples: 6319
Test samples: 1358


In [2]:
# DATASET CLASS AND FUNCTIONS
class WasteJSONDataset(Dataset):
    def __init__(self, root_dir, file_list, transform=None):
        self.root_dir = root_dir
        self.file_list = file_list
        self.transform = transform
        self.class_to_idx = {cat: i for i, cat in enumerate(CATEGORIES)}

    def __len__(self): 
        return len(self.file_list)

    def __getitem__(self, idx):
        rel_path = self.file_list[idx]
        img_path = os.path.join(self.root_dir, rel_path)
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            image = Image.new('RGB', (512, 512), (0, 0, 0))
            
        label = self.class_to_idx[rel_path.split('/')[0]]
        if self.transform: 
            image = self.transform(image)
        return image, label

def get_dataloaders(version, batch_size=32):
    """Generate data loaders with Augmentation and Resizing to 224"""
    if version == 'raw':
        data_path = os.path.join(DATA_DIR, "Cropped")
    else:
        data_path = os.path.join(DATA_DIR, f"processed_{version}")
    
    # IMPROVEMENT 1: Stronger Augmentation for Training
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),  # Resize to standard ResNet size (was 512)
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    # Clean transform for Validation
    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    train_ds = WasteJSONDataset(data_path, split['train'], transform=train_transform)
    test_ds = WasteJSONDataset(data_path, split['test'], transform=test_transform)
    
    # IMPROVEMENT 2: Enable multi-processing and pinned memory
    # Note: On Windows, if num_workers > 0 crashes, set it back to 0.
    return {
        'train': DataLoader(train_ds, batch_size=batch_size, shuffle=True, 
                          num_workers=4, pin_memory=True, persistent_workers=True),
        'test': DataLoader(test_ds, batch_size=batch_size, shuffle=False, 
                         num_workers=4, pin_memory=True, persistent_workers=True)
    }, len(train_ds), len(test_ds)

def train_model(model, loaders, criterion, optimizer, num_epochs=5):
    """Train the model"""
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        progress_bar = tqdm(loaders['train'], desc=f'Epoch {epoch+1}/{num_epochs}')
        
        for inputs, labels in progress_bar:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            progress_bar.set_postfix({'Loss': f'{loss.item():.4f}'})
        
        epoch_loss = running_loss / len(loaders['train'].dataset)
        print(f"Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f}")
    
    return model

def evaluate_and_record(model, loaders, version):
    """Evaluate model and return metrics"""
    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for inputs, labels in tqdm(loaders['test'], desc='Evaluating'):
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    metrics = {
        'Model': 'ResNet-50',
        'Preprocessing': version,
        'Accuracy': accuracy_score(all_labels, all_preds),
        'Precision': precision_score(all_labels, all_preds, average='macro', zero_division=0),
        'Recall': recall_score(all_labels, all_preds, average='macro', zero_division=0),
        'F1-score': f1_score(all_labels, all_preds, average='macro', zero_division=0)
    }
    return metrics

def run_resnet_experiment(version):
    print(f"\n{'='*60}")
    print(f"🚀 ResNet-50 Experiment: {version.upper()}")
    print(f"{'='*60}")
    
    try:
        # Increase batch size slightly since image size is smaller (224 vs 512)
        loaders, train_size, test_size = get_dataloaders(version=version, batch_size=64)
        print(f"✅ Data loaded (Resized to 224x224)")
        
        # Initialize Pretrained ResNet50
        model = models.resnet50(weights='IMAGENET1K_V2')
        
        # Fine-tuning strategy:
        # Optional: Freeze early layers to speed up training further
        # for param in model.parameters():
        #     param.requires_grad = False
            
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, 4)
        model = model.to(DEVICE)
        
        criterion = nn.CrossEntropyLoss()
        
        # IMPROVEMENT 3: Use AdamW with Weight Decay
        optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
        
        # IMPROVEMENT 4: Learning Rate Scheduler
        # Decreases LR if loss stops improving
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)
        
        print("Starting training...")
        # Pass scheduler to train_model if you modify that function, 
        # otherwise step it inside the loop manually or ignore for now.
        trained_model = train_model(model, loaders, criterion, optimizer, num_epochs=10) # Increased epochs
        
        # ... (Rest of the saving logic remains the same) ...
        print(f"\n📊 Results for {version}:")
        for key, value in res.items():
            if key not in ['Model', 'Preprocessing']:
                print(f"  {key}: {value:.4f}")
        
        # Save individual result
        result_path = os.path.join(DATA_DIR, f"resnet50_{version}_results.csv")
        pd.DataFrame([res]).to_csv(result_path, index=False)
        print(f"✅ Results saved to {result_path}")
        
        return res
        
    except Exception as e:
        print(f"❌ Error processing {version}: {e}")
        return None

print("✅ Setup complete!")

✅ Setup complete!


In [ ]:
# 🔥 EXPERIMENT 1: RAW
torch.cuda.empty_cache()
gc.collect()

result_raw = run_resnet_experiment('raw')

# Clear memory
torch.cuda.empty_cache()
gc.collect()
print("\n🧹 Memory cleared after RAW experiment")


🚀 ResNet-50 Experiment: RAW
✅ Data loaded (Resized to 224x224)
Starting training...


Epoch 1/10:   0%|          | 0/99 [00:00<?, ?it/s]

In [ ]:
# 🔥 EXPERIMENT 2: UNIFIED
torch.cuda.empty_cache()
gc.collect()

result_unified = run_resnet_experiment('unified')

# Clear memory
torch.cuda.empty_cache()
gc.collect()
print("\n🧹 Memory cleared after UNIFIED experiment")

In [ ]:
# 🔥 EXPERIMENT 3: CUSTOMIZED
torch.cuda.empty_cache()
gc.collect()

result_customized = run_resnet_experiment('customized')

# Clear memory
torch.cuda.empty_cache()
gc.collect()
print("\n🧹 Memory cleared after CUSTOMIZED experiment")

In [ ]:
# 📊 CONSOLIDATE ALL RESULTS
all_results = []
for result in [result_raw, result_unified, result_customized]:
    if result is not None:
        all_results.append(result)

if all_results:
    # Save consolidated results
    consolidated_path = os.path.join(DATA_DIR, "resnet50_all_results.csv")
    df_results = pd.DataFrame(all_results)
    df_results.to_csv(consolidated_path, index=False)
    
    print(f"\n✅ Consolidated results saved to {consolidated_path}")
    print(f"\n📊 FINAL RESNET-50 SUMMARY:")
    print("="*60)
    print(df_results.to_string(index=False))
    
    # Find best result
    best_idx = df_results['Accuracy'].idxmax()
    best_result = df_results.iloc[best_idx]
    print(f"\n🏆 BEST RESNET-50: {best_result['Preprocessing']} (Accuracy: {best_result['Accuracy']:.4f})")
else:
    print("❌ No results to consolidate")

print("\n🏁 ResNet-50 experiments completed!")